# 算法嵌套与混合策略

在实际优化问题中，单一算法往往难以应对所有场景。NSGABLACK框架支持多种算法组合策略：

1. **算法嵌套** - 一个算法作为另一个算法的组件
2. **并行执行** - 多算法同时运行
3. **序列执行** - 算法按顺序执行
4. **协同进化** - 算法间信息共享

In [ ]:
# 导入必要的库
import numpy as np
import matplotlib.pyplot as plt
from typing import List, Callable, Tuple, Any
import time
from abc import ABC, abstractmethod
import concurrent.futures

# 设置中文显示
plt.rcParams['font.sans-serif'] = ['SimHei']
plt.rcParams['axes.unicode_minus'] = False

print("环境准备完成")
print(f"NumPy版本: {np.__version__}")

## 算法基础接口

In [ ]:
class BaseOptimizer(ABC):
    """优化器基类"""
    
    def __init__(self, name="BaseOptimizer", seed=None):
        self.name = name
        self.seed = seed
        self.history = []
        self.best_solution = None
        self.best_fitness = float('inf')
    
    @abstractmethod
    def optimize(self, objective_func, n_var, bounds, **kwargs):
        """优化接口"""
        pass

class SimpleRandomSearch(BaseOptimizer):
    """简单随机搜索"""
    
    def __init__(self, max_evals=1000, seed=None):
        super().__init__("RandomSearch", seed)
        self.max_evals = max_evals
        if seed is not None:
            np.random.seed(seed)
    
    def optimize(self, objective_func, n_var, bounds, **kwargs):
        for _ in range(self.max_evals):
            x = np.array([np.random.uniform(b[0], b[1]) for b in bounds])
            f = objective_func(x)
            
            if f < self.best_fitness:
                self.best_fitness = f
                self.best_solution = x.copy()
            
            self.history.append(self.best_fitness)
        
        return self.best_solution, self.best_fitness

class SimpleLocalSearch(BaseOptimizer):
    """简单局部搜索"""
    
    def __init__(self, max_iter=100, step_size=0.1, seed=None):
        super().__init__("LocalSearch", seed)
        self.max_iter = max_iter
        self.step_size = step_size
        if seed is not None:
            np.random.seed(seed)
    
    def optimize(self, objective_func, n_var, bounds, initial_x=None, **kwargs):
        # 初始化解
        if initial_x is None:
            x = np.array([np.random.uniform(b[0], b[1]) for b in bounds])
        else:
            x = initial_x.copy()
        
        self.best_solution = x.copy()
        self.best_fitness = objective_func(x)
        
        for _ in range(self.max_iter):
            improved = False
            
            for i in range(n_var):
                # 尝试正方向
                x_new = x.copy()
                x_new[i] += self.step_size * (bounds[i][1] - bounds[i][0])
                x_new[i] = min(x_new[i], bounds[i][1])
                
                f_new = objective_func(x_new)
                if f_new < self.best_fitness:
                    x = x_new
                    self.best_fitness = f_new
                    self.best_solution = x.copy()
                    improved = True
                    continue
                
                # 尝试负方向
                x_new = x.copy()
                x_new[i] -= self.step_size * (bounds[i][1] - bounds[i][0])
                x_new[i] = max(x_new[i], bounds[i][0])
                
                f_new = objective_func(x_new)
                if f_new < self.best_fitness:
                    x = x_new
                    self.best_fitness = f_new
                    self.best_solution = x.copy()
                    improved = True
            
            if not improved:
                break
            
            self.history.append(self.best_fitness)
        
        return self.best_solution, self.best_fitness

print("\n基础优化器类已定义")

## 策略1：算法嵌套

In [ ]:
class NestedOptimizer:
    """嵌套优化器 - 外层算法提供初始解，内层算法精细优化"""
    
    def __init__(self, outer_optimizer, inner_optimizer, 
                 outer_to_inner_ratio=0.1, seed=None):
        self.outer_optimizer = outer_optimizer
        self.inner_optimizer = inner_optimizer
        self.outer_to_inner_ratio = outer_to_inner_ratio
        self.seed = seed
        
        # 记录嵌套过程
        self.outer_history = []
        self.inner_history = []
        self.best_solution = None
        self.best_fitness = float('inf')
        
        print(f"创建嵌套优化器")
        print(f"  外层算法: {outer_optimizer.__class__.__name__}")
        print(f"  内层算法: {inner_optimizer.__class__.__name__}")
        print(f"  外/内迭代比: {outer_to_inner_ratio}")
    
    def optimize(self, objective_func, n_var, bounds, **kwargs):
        print("\n开始嵌套优化...")
        
        # 第一阶段：外层算法全局搜索
        print("\n阶段1: 外层全局搜索")
        outer_x, outer_f = self.outer_optimizer.optimize(
            objective_func, n_var, bounds
        )
        self.outer_history = self.outer_optimizer.history
        
        print(f"外层最优解: {outer_x}, 适应度: {outer_f:.4f}")
        
        # 第二阶段：内层算法局部精化
        print("\n阶段2: 内层局部精化")
        inner_x, inner_f = self.inner_optimizer.optimize(
            objective_func, n_var, bounds, initial_x=outer_x
        )
        self.inner_history = self.inner_optimizer.history
        
        print(f"内层最优解: {inner_x}, 适应度: {inner_f:.4f}")
        
        self.best_solution = inner_x
        self.best_fitness = inner_f
        
        improvement = (outer_f - inner_f) / abs(outer_f) * 100
        print(f"\n总体改进: {improvement:.2f}%")
        
        return self.best_solution, self.best_fitness
    
    def plot_optimization_process(self):
        """绘制嵌套优化过程"""
        if not self.outer_history or not self.inner_history:
            print("没有优化历史可绘制")
            return
        
        fig, axes = plt.subplots(1, 2, figsize=(15, 5))
        
        # 外层优化过程
        ax1 = axes[0]
        ax1.plot(self.outer_history, 'b-', linewidth=2, label='外层优化')
        ax1.set_xlabel('迭代次数')
        ax1.set_ylabel('最优适应度')
        ax1.set_title('外层全局搜索')
        ax1.grid(True, alpha=0.3)
        ax1.legend()
        
        # 内层优化过程
        ax2 = axes[1]
        ax2.plot(self.inner_history, 'r-', linewidth=2, label='内层精化')
        ax2.set_xlabel('迭代次数')
        ax2.set_ylabel('最优适应度')
        ax2.set_title('内层局部精化')
        ax2.grid(True, alpha=0.3)
        ax2.legend()
        
        plt.tight_layout()
        plt.show()

# 测试嵌套优化
print("\n测试嵌套优化策略：")
print("=" * 50)

# 定义测试函数
def test_function(x):
    """多峰测试函数"""
    return (x[0]**2 + x[1]**2) * np.exp(-(x[0]**2 + x[1]**2) / 10)

# 创建嵌套优化器
outer = SimpleRandomSearch(max_evals=500, seed=42)
inner = SimpleLocalSearch(max_iter=100, step_size=0.05, seed=42)

nested_opt = NestedOptimizer(outer, inner, seed=42)
best_x, best_f = nested_opt.optimize(
    test_function, 
    n_var=2, 
    bounds=[(-5, 5), (-5, 5)]
)

# 绘制过程
nested_opt.plot_optimization_process()

## 策略2：并行执行

In [ ]:
class ParallelOptimizer:
    """并行优化器 - 多算法同时运行，取最优结果"""
    
    def __init__(self, optimizers, n_workers=None):
        self.optimizers = optimizers
        self.n_workers = n_workers or len(optimizers)
        self.results = {}
        
        print(f"创建并行优化器")
        print(f"  算法数量: {len(optimizers)}")
        print(f"  并行工作数: {self.n_workers}")
        for i, opt in enumerate(optimizers):
            print(f"  算法{i+1}: {opt.__class__.__name__}")
    
    def _run_optimizer(self, optimizer_id, optimizer, objective_func, n_var, bounds):
        """运行单个优化器"""
        start_time = time.time()
        best_x, best_f = optimizer.optimize(objective_func, n_var, bounds)
        elapsed_time = time.time() - start_time
        
        return {
            'optimizer_id': optimizer_id,
            'optimizer_name': optimizer.__class__.__name__,
            'best_x': best_x,
            'best_f': best_f,
            'history': optimizer.history,
            'elapsed_time': elapsed_time
        }
    
    def optimize(self, objective_func, n_var, bounds, **kwargs):
        print("\n开始并行优化...")
        
        # 并行执行所有优化器
        with concurrent.futures.ThreadPoolExecutor(max_workers=self.n_workers) as executor:
            futures = []
            
            for i, optimizer in enumerate(self.optimizers):
                future = executor.submit(
                    self._run_optimizer,
                    i, optimizer, objective_func, n_var, bounds
                )
                futures.append(future)
            
            # 收集结果
            results = []
            for future in concurrent.futures.as_completed(futures):
                result = future.result()
                results.append(result)
                self.results[result['optimizer_id']] = result
        
        # 找到最优结果
        best_result = min(results, key=lambda x: x['best_f'])
        
        print("\n并行优化完成！")
        print("=" * 50)
        for res in results:
            print(f"\n{res['optimizer_name']}:")
            print(f"  最优适应度: {res['best_f']:.4f}")
            print(f"  运行时间: {res['elapsed_time']:.2f}秒")
        
        print(f"\n最佳算法: {best_result['optimizer_name']}")
        print(f"全局最优适应度: {best_result['best_f']:.4f}")
        
        return best_result['best_x'], best_result['best_f']
    
    def plot_comparison(self):
        """绘制各算法性能对比"""
        if not self.results:
            print("没有结果可绘制")
            return
        
        fig, axes = plt.subplots(2, 2, figsize=(15, 10))
        
        # 收敛曲线对比
        ax1 = axes[0, 0]
        for opt_id, result in self.results.items():
            if result['history']:
                ax1.plot(result['history'], label=result['optimizer_name'], linewidth=2)
        ax1.set_xlabel('迭代次数')
        ax1.set_ylabel('最优适应度')
        ax1.set_title('收敛曲线对比')
        ax1.legend()
        ax1.grid(True, alpha=0.3)
        ax1.set_yscale('log')
        
        # 最终结果对比
        ax2 = axes[0, 1]
        names = [r['optimizer_name'] for r in self.results.values()]
        values = [r['best_f'] for r in self.results.values()]
        bars = ax2.bar(names, values, alpha=0.7)
        ax2.set_ylabel('最优适应度')
        ax2.set_title('最终结果对比')
        ax2.grid(True, alpha=0.3, axis='y')
        plt.setp(ax2.get_xticklabels(), rotation=45)
        
        # 添加数值标签
        for bar, val in zip(bars, values):
            ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height(), 
                    f'{val:.2f}', ha='center', va='bottom')
        
        # 运行时间对比
        ax3 = axes[1, 0]
        times = [r['elapsed_time'] for r in self.results.values()]
        bars = ax3.bar(names, times, alpha=0.7, color='orange')
        ax3.set_ylabel('运行时间(秒)')
        ax3.set_title('运行时间对比')
        ax3.grid(True, alpha=0.3, axis='y')
        plt.setp(ax3.get_xticklabels(), rotation=45)
        
        # 添加数值标签
        for bar, val in zip(bars, times):
            ax3.text(bar.get_x() + bar.get_width()/2, bar.get_height(), 
                    f'{val:.2f}', ha='center', va='bottom')
        
        # 性能散点图（适应度vs时间）
        ax4 = axes[1, 1]
        for opt_id, result in self.results.items():
            ax4.scatter(result['elapsed_time'], result['best_f'], 
                       s=100, alpha=0.7, label=result['optimizer_name'])
        ax4.set_xlabel('运行时间(秒)')
        ax4.set_ylabel('最优适应度')
        ax4.set_title('性能散点图')
        ax4.legend()
        ax4.grid(True, alpha=0.3)
        ax4.set_xscale('log')
        ax4.set_yscale('log')
        
        plt.tight_layout()
        plt.show()

# 测试并行优化
print("\n测试并行优化策略：")
print("=" * 50)

# 创建多个优化器
opt1 = SimpleRandomSearch(max_evals=500, seed=42)
opt2 = SimpleRandomSearch(max_evals=1000, seed=123)
opt3 = SimpleLocalSearch(max_iter=200, step_size=0.1, seed=456)

# 创建并行优化器
parallel_opt = ParallelOptimizer([opt1, opt2, opt3], n_workers=3)
best_x, best_f = parallel_opt.optimize(
    test_function,
    n_var=2,
    bounds=[(-5, 5), (-5, 5)]
)

# 绘制对比
parallel_opt.plot_comparison()

## 策略3：序列执行

In [ ]:
class SequentialOptimizer:
    """序列优化器 - 算法按顺序执行，传递最优解"""
    
    def __init__(self, optimizers, transfer_strategy='best'):
        self.optimizers = optimizers
        self.transfer_strategy = transfer_strategy
        self.stage_results = []
        
        print(f"创建序列优化器")
        print(f"  算法序列: {[opt.__class__.__name__ for opt in optimizers]}")
        print(f"  传递策略: {transfer_strategy}")
    
    def optimize(self, objective_func, n_var, bounds, **kwargs):
        print("\n开始序列优化...")
        
        current_solution = None
        current_fitness = float('inf')
        
        for stage, optimizer in enumerate(self.optimizers):
            print(f"\n阶段 {stage + 1}: {optimizer.__class__.__name__}")
            print("-" * 30)
            
            # 传递初始解（除了第一个算法）
            if stage > 0 and self.transfer_strategy == 'best':
                print(f"初始解: {current_solution}, 适应度: {current_fitness:.4f}")
                
            # 运行当前算法
            if hasattr(optimizer, 'optimize'):
                # 如果算法支持初始解
                try:
                    best_x, best_f = optimizer.optimize(
                        objective_func, n_var, bounds, 
                        initial_x=current_solution
                    )
                except:
                    best_x, best_f = optimizer.optimize(
                        objective_func, n_var, bounds
                    )
            else:
                raise ValueError(f"优化器 {optimizer} 必须实现 optimize 方法")
            
            # 更新当前最优解
            if best_f < current_fitness:
                current_solution = best_x
                current_fitness = best_f
                print(f"改进! 新适应度: {current_fitness:.4f}")
            else:
                print(f"无改进，保持适应度: {current_fitness:.4f}")
            
            # 记录阶段结果
            self.stage_results.append({
                'stage': stage + 1,
                'optimizer': optimizer.__class__.__name__,
                'best_x': best_x,
                'best_f': best_f,
                'improved': best_f < current_fitness if stage > 0 else True,
                'history': getattr(optimizer, 'history', [])
            })
        
        print("\n序列优化完成！")
        print(f"最终最优适应度: {current_fitness:.4f}")
        
        return current_solution, current_fitness
    
    def plot_progression(self):
        """绘制序列优化进展"""
        if not self.stage_results:
            print("没有结果可绘制")
            return
        
        fig, axes = plt.subplots(2, 2, figsize=(15, 10))
        
        # 阶段适应度变化
        ax1 = axes[0, 0]
        stages = range(1, len(self.stage_results) + 1)
        fitnesses = [r['best_f'] for r in self.stage_results]
        ax1.plot(stages, fitnesses, 'bo-', linewidth=2, markersize=8)
        ax1.set_xlabel('阶段')
        ax1.set_ylabel('最优适应度')
        ax1.set_title('各阶段最优适应度')
        ax1.grid(True, alpha=0.3)
        ax1.set_yscale('log')
        
        # 改进标记
        for i, (stage, result) in enumerate(self.stage_results.items()):
            color = 'green' if result['improved'] else 'red'
            ax1.scatter(i+1, result['best_f'], s=100, c=color, 
                      marker='o' if result['improved'] else 'x')
        
        # 各算法收敛过程
        ax2 = axes[0, 1]
        total_evals = 0
        for stage, result in self.stage_results.items():
            if result['history']:
                evals = range(total_evals, total_evals + len(result['history']))
                ax2.plot(evals, result['history'], 
                        label=f"阶段{stage}: {result['optimizer']}", 
                        linewidth=2)
                total_evals += len(result['history'])
        ax2.set_xlabel('总评估次数')
        ax2.set_ylabel('适应度')
        ax2.set_title('完整收敛过程')
        ax2.legend()
        ax2.grid(True, alpha=0.3)
        ax2.set_yscale('log')
        
        # 改进幅度
        ax3 = axes[1, 0]
        improvements = []
        labels = []
        for i, (stage, result) in enumerate(self.stage_results.items()):
            if i > 0:
                prev_f = self.stage_results[i-1]['best_f']
                curr_f = result['best_f']
                improvement = (prev_f - curr_f) / abs(prev_f) * 100
                improvements.append(improvement)
                labels.append(result['optimizer'])
                
                color = 'green' if improvement > 0 else 'red'
                ax3.bar(labels[-1], improvement, alpha=0.7, color=color)
        
        ax3.set_ylabel('改进百分比(%)')
        ax3.set_title('各算法改进幅度')
        ax3.grid(True, alpha=0.3, axis='y')
        plt.setp(ax3.get_xticklabels(), rotation=45)
        ax3.axhline(y=0, color='black', linestyle='-', alpha=0.3)
        
        # 阶段时间分配
        ax4 = axes[1, 1]
        if all('history' in r for r in self.stage_results.values()):
            eval_counts = [len(r['history']) for r in self.stage_results.values()]
            labels = [r['optimizer'] for r in self.stage_results.values()]
            
            colors = plt.cm.viridis(np.linspace(0, 1, len(labels)))
            wedges, texts, autotexts = ax4.pie(
                eval_counts, labels=labels, autopct='%1.1f%%',
                colors=colors, startangle=90
            )
            ax4.set_title('评估次数分配')
        
        plt.tight_layout()
        plt.show()

# 测试序列优化
print("\n测试序列优化策略：")
print("=" * 50)

# 创建优化器序列
seq_opt1 = SimpleRandomSearch(max_evals=300, seed=42)  # 快速全局搜索
seq_opt2 = SimpleLocalSearch(max_iter=100, step_size=0.2, seed=42)  # 中等精化
seq_opt3 = SimpleLocalSearch(max_iter=50, step_size=0.05, seed=42)  # 精细搜索

# 创建序列优化器
sequential_opt = SequentialOptimizer([seq_opt1, seq_opt2, seq_opt3])
best_x, best_f = sequential_opt.optimize(
    test_function,
    n_var=2,
    bounds=[(-5, 5), (-5, 5)]
)

# 绘制进展
sequential_opt.plot_progression()

## 策略4：协同进化

In [ ]:
class CooperativeOptimizer:
    """协同进化优化器 - 多算法信息共享"""
    
    def __init__(self, optimizers, info_exchange_rate=0.1, 
                 exchange_strategy='best', max_iter=100):
        self.optimizers = optimizers
        self.info_exchange_rate = info_exchange_rate
        self.exchange_strategy = exchange_strategy
        self.max_iter = max_iter
        
        # 信息共享
        self.shared_best = None
        self.shared_fitness = float('inf')
        self.optimization_history = {i: [] for i in range(len(optimizers))}
        
        print(f"创建协同进化优化器")
        print(f"  算法数量: {len(optimizers)}")
        print(f"  信息交换率: {info_exchange_rate}")
        print(f"  交换策略: {exchange_strategy}")
    
    def _exchange_information(self, optimizer_id, optimizer):
        """信息交换"""
        if np.random.random() < self.info_exchange_rate:
            if self.exchange_strategy == 'best' and self.shared_best is not None:
                # 使用共享的最优解
                return self.shared_best
            elif self.exchange_strategy == 'random':
                # 随机选择另一个算法的解
                other_ids = [i for i in range(len(self.optimizers)) if i != optimizer_id]
                if other_ids:
                    other_id = np.random.choice(other_ids)
                    other_opt = self.optimizers[other_id]
                    if hasattr(other_opt, 'best_solution') and other_opt.best_solution is not None:
                        return other_opt.best_solution
        return None
    
    def _update_shared_info(self):
        """更新共享信息"""
        best_solutions = []
        
        for opt in self.optimizers:
            if hasattr(opt, 'best_solution') and opt.best_solution is not None:
                best_solutions.append((opt.best_solution, opt.best_fitness))
        
        if best_solutions:
            # 找到全局最优
            global_best = min(best_solutions, key=lambda x: x[1])
            if global_best[1] < self.shared_fitness:
                self.shared_best = global_best[0].copy()
                self.shared_fitness = global_best[1]
    
    def optimize(self, objective_func, n_var, bounds, **kwargs):
        print("\n开始协同进化优化...")
        
        # 初始化
        for i, optimizer in enumerate(self.optimizers):
            optimizer.best_solution = None
            optimizer.best_fitness = float('inf')
            optimizer.history = []
        
        # 主循环
        for iteration in range(self.max_iter):
            # 每个算法执行一步
            for i, optimizer in enumerate(self.optimizers):
                # 尝试信息交换
                shared_solution = self._exchange_information(i, optimizer)
                
                # 执行优化步骤
                if hasattr(optimizer, '_step'):
                    # 如果有单步方法
                    optimizer._step(objective_func, n_var, bounds, 
                                 shared_solution=shared_solution)
                else:
                    # 否则执行一次完整优化
                    try:
                        optimizer.optimize(objective_func, n_var, bounds, 
                                         max_evals=1, initial_x=shared_solution)
                    except:
                        pass
                
                # 记录历史
                if hasattr(optimizer, 'best_fitness'):
                    self.optimization_history[i].append(optimizer.best_fitness)
            
            # 更新共享信息
            self._update_shared_info()
            
            # 输出进度
            if (iteration + 1) % 20 == 0:
                print(f"  迭代 {iteration+1}: 共享最优 = {self.shared_fitness:.4f}")
        
        print(f"\n协同进化完成！")
        print(f"共享最优适应度: {self.shared_fitness:.4f}")
        
        return self.shared_best, self.shared_fitness
    
    def plot_cooperation(self):
        """绘制协同进化过程"""
        if not any(self.optimization_history.values()):
            print("没有优化历史可绘制")
            return
        
        fig, axes = plt.subplots(2, 2, figsize=(15, 10))
        
        # 各算法收敛曲线
        ax1 = axes[0, 0]
        for i, history in self.optimization_history.items():
            if history:
                ax1.plot(history, label=f"算法{i+1}", linewidth=2, alpha=0.8)
        ax1.set_xlabel('迭代次数')
        ax1.set_ylabel('适应度')
        ax1.set_title('各算法收敛过程')
        ax1.legend()
        ax1.grid(True, alpha=0.3)
        ax1.set_yscale('log')
        
        # 算法性能对比
        ax2 = axes[0, 1]
        final_fitnesses = []
        labels = []
        
        for i, (opt, history) in enumerate(zip(self.optimizers, self.optimization_history.values())):
            if history:
                final_fitnesses.append(history[-1])
                labels.append(f"算法{i+1}")
        
        if final_fitnesses:
            bars = ax2.bar(labels, final_fitnesses, alpha=0.7)
            ax2.set_ylabel('最终适应度')
            ax2.set_title('各算法最终性能')
            ax2.grid(True, alpha=0.3, axis='y')
            
            # 添加数值标签
            for bar, val in zip(bars, final_fitnesses):
                ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height(), 
                        f'{val:.2f}', ha='center', va='bottom')
        
        # 收敛速度对比
        ax3 = axes[1, 0]
        convergence_iters = []
        
        for history in self.optimization_history.values():
            if history:
                # 找到达到最终性能90%的迭代次数
                target = history[0] * 0.9
                for i, f in enumerate(history):
                    if f <= target:
                        convergence_iters.append(i)
                        break
                else:
                    convergence_iters.append(len(history))
        
        if convergence_iters:
            bars = ax3.bar(labels[:len(convergence_iters)], convergence_iters, 
                         alpha=0.7, color='orange')
            ax3.set_ylabel('收敛迭代次数')
            ax3.set_title('收敛速度对比')
            ax3.grid(True, alpha=0.3, axis='y')
        
        # 协作效果
        ax4 = axes[1, 1]
        if all(self.optimization_history.values()):
            # 计算平均性能
            min_len = min(len(h) for h in self.optimization_history.values())
            avg_performance = []
            best_performance = []
            
            for i in range(min_len):
                values = [h[i] for h in self.optimization_history.values()]
                avg_performance.append(np.mean(values))
                best_performance.append(min(values))
            
            iterations = range(min_len)
            ax4.plot(iterations, avg_performance, 'b-', 
                    linewidth=2, label='平均性能')
            ax4.plot(iterations, best_performance, 'r-', 
                    linewidth=2, label='最佳性能')
            ax4.set_xlabel('迭代次数')
            ax4.set_ylabel('适应度')
            ax4.set_title('协作效果')
            ax4.legend()
            ax4.grid(True, alpha=0.3)
            ax4.set_yscale('log')
        
        plt.tight_layout()
        plt.show()

# 测试协同进化
print("\n测试协同进化策略：")
print("=" * 50)

# 创建简单的进化算法用于协同
class SimpleEvolution:
    def __init__(self, pop_size=20, mutation_rate=0.1, seed=None):
        self.pop_size = pop_size
        self.mutation_rate = mutation_rate
        self.seed = seed
        if seed is not None:
            np.random.seed(seed)
    
    def optimize(self, objective_func, n_var, bounds, **kwargs):
        # 初始化种群
        population = np.array([
            [np.random.uniform(b[0], b[1]) for b in bounds] 
            for _ in range(self.pop_size)
        ])
        
        for _ in range(100):
            # 评估
            fitnesses = np.array([objective_func(ind) for ind in population])
            
            # 选择
            best_idx = np.argmin(fitnesses)
            self.best_solution = population[best_idx]
            self.best_fitness = fitnesses[best_idx]
            
            # 变异
            for i in range(self.pop_size):
                if np.random.random() < self.mutation_rate:
                    j = np.random.randint(0, n_var)
                    population[i][j] = np.random.uniform(bounds[j][0], bounds[j][1])
        
        return self.best_solution, self.best_fitness

# 创建协同优化器
coop_opt1 = SimpleEvolution(pop_size=20, mutation_rate=0.2, seed=42)
coop_opt2 = SimpleEvolution(pop_size=30, mutation_rate=0.1, seed=123)
coop_opt3 = SimpleEvolution(pop_size=25, mutation_rate=0.15, seed=456)

cooperative_opt = CooperativeOptimizer(
    [coop_opt1, coop_opt2, coop_opt3],
    info_exchange_rate=0.2,
    exchange_strategy='best',
    max_iter=100
)

best_x, best_f = cooperative_opt.optimize(
    test_function,
    n_var=2,
    bounds=[(-5, 5), (-5, 5)]
)

# 绘制协作过程
cooperative_opt.plot_cooperation()

## 混合策略对比

In [ ]:
# 对比不同混合策略
print("\n对比不同混合策略性能：")
print("=" * 60)

# 定义复杂的测试函数
def complex_function(x):
    """复杂多峰函数"""
    term1 = np.sin(x[0] + x[1])**2 + np.sin(x[0] - x[1])**2
    term2 = (x[0]**2 + x[1]**2) / 10
    term3 = np.exp(-((x[0]-2)**2 + (x[1]+2)**2) / 5)
    return term1 + term2 - term3

# 策略1：嵌套优化
print("\n1. 嵌套优化策略：")
nested_outer = SimpleRandomSearch(max_evals=200, seed=42)
nested_inner = SimpleLocalSearch(max_iter=50, step_size=0.1, seed=42)
nested = NestedOptimizer(nested_outer, nested_inner, seed=42)
start = time.time()
nested_x, nested_f = nested.optimize(complex_function, 2, [(-5, 5), (-5, 5)])
nested_time = time.time() - start

# 策略2：并行优化
print("\n2. 并行优化策略：")
para_opt1 = SimpleRandomSearch(max_evals=300, seed=42)
para_opt2 = SimpleLocalSearch(max_iter=100, step_size=0.1, seed=123)
parallel = ParallelOptimizer([para_opt1, para_opt2], n_workers=2)
start = time.time()
parallel_x, parallel_f = parallel.optimize(complex_function, 2, [(-5, 5), (-5, 5)])
parallel_time = time.time() - start

# 策略3：序列优化
print("\n3. 序列优化策略：")
seq_opt1 = SimpleRandomSearch(max_evals=200, seed=42)
seq_opt2 = SimpleLocalSearch(max_iter=50, step_size=0.1, seed=42)
sequential = SequentialOptimizer([seq_opt1, seq_opt2])
start = time.time()
sequential_x, sequential_f = sequential.optimize(complex_function, 2, [(-5, 5), (-5, 5)])
sequential_time = time.time() - start

# 策略4：单一算法（基准）
print("\n4. 单一算法（基准）：")
single_opt = SimpleRandomSearch(max_evals=500, seed=42)
start = time.time()
single_x, single_f = single_opt.optimize(complex_function, 2, [(-5, 5), (-5, 5)])
single_time = time.time() - start

# 结果对比
results = [
    ('嵌套优化', nested_f, nested_time),
    ('并行优化', parallel_f, parallel_time),
    ('序列优化', sequential_f, sequential_time),
    ('单一算法', single_f, single_time)
]

print("\n" + "="*60)
print("策略对比结果：")
print("="*60)
print(f"{'策略':<15} {'最优适应度':<15} {'运行时间(秒)':<15} {'效率比':<15}")
print("-"*60)

for name, fitness, time_used in results:
    efficiency = fitness / time_used
    print(f"{name:<15} {fitness:<15.4f} {time_used:<15.2f} {efficiency:<15.4f}")

# 找到最佳策略
best_strategy = min(results, key=lambda x: x[1])
print(f"\n最佳策略: {best_strategy[0]}")
print(f"最优适应度: {best_strategy[1]:.4f}")

# 可视化对比
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 适应度对比
names = [r[0] for r in results]
fitnesses = [r[1] for r in results]
colors = ['gold', 'silver', 'orange', 'gray']
bars = axes[0].bar(names, fitnesses, alpha=0.7, color=colors)
axes[0].set_ylabel('最优适应度')
axes[0].set_title('各策略最优适应度对比')
axes[0].grid(True, alpha=0.3, axis='y')
plt.setp(axes[0].get_xticklabels(), rotation=45)

# 时间对比
times = [r[2] for r in results]
bars = axes[1].bar(names, times, alpha=0.7, color=colors)
axes[1].set_ylabel('运行时间(秒)')
axes[1].set_title('各策略运行时间对比')
axes[1].grid(True, alpha=0.3, axis='y')
plt.setp(axes[1].get_xticklabels(), rotation=45)

# 效率散点图
axes[2].scatter(times, fitnesses, s=100, alpha=0.7, c=colors)
for i, (name, f, t) in enumerate(results):
    axes[2].annotate(name, (t, f), xytext=(5, 5), 
                    textcoords='offset points')
axes[2].set_xlabel('运行时间(秒)')
axes[2].set_ylabel('最优适应度')
axes[2].set_title('效率分析')
axes[2].grid(True, alpha=0.3)

# 添加理想区域标注
axes[2].axhspan(min(fitnesses), min(fitnesses)*1.1, 
                 color='green', alpha=0.1, label='理想区域')
axes[2].axvspan(0, max(times)*0.3, 
                 color='green', alpha=0.1)

plt.tight_layout()
plt.show()

print("\n策略选择建议：")
print("- 嵌套优化: 适合需要全局搜索+局部精化的问题")
print("- 并行优化: 适合计算资源充足，需要快速得到结果")
print("- 序列优化: 适合不同算法有明确分工的场景")
print("- 单一算法: 适合简单问题或作为基准对比")

## SGABLACK框架的算法集成能力

SGABLACK框架设计原则之一是**可扩展性**，能够轻松集成各种优化算法：

### 1. 统一接口
所有优化器都继承自基类，确保一致性

### 2. 偏置系统集成
任何算法都可以利用偏置系统增强性能

### 3. 模块化设计
算法可独立开发和测试

### 4. 混合策略支持
天然支持各种算法组合方式

### 未来可扩展的算法：
- **MOEA/D** - 多目标进化算法基于分解
- **PSO** - 粒子群优化
- **DE** - 差分进化
- **ACO** - 蚁群优化
- **SA** - 模拟退火
- **GA** - 遗传算法

每种算法都可以与偏置系统结合，形成独特的优化策略。

## 总结

算法嵌套与混合策略要点：

1. **嵌套优化** - 外层全局探索，内层局部精化
2. **并行执行** - 多算法同时运行，充分利用资源
3. **序列执行** - 算法分工协作，逐步改进
4. **协同进化** - 信息共享，集体智慧

选择策略的考虑因素：
- 问题复杂度
- 计算资源
- 时间要求
- 算法特性

SGABLACK框架提供了灵活的基础设施，支持各种算法组合策略，让用户能够根据具体需求选择最适合的优化方案。